In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, max as spark_max, min as spark_min, count, sum as spark_sum, 
    avg, datediff, current_date, when, lit, round as spark_round,
    ntile, concat_ws
)
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql.dataframe import DataFrame
from pyspark.errors import AnalysisException
from delta.tables import *
from typing import Dict, List
from utils.functions.prepare_system_columns import prepare_system_dataframe
from utils.functions.upsert import upsert_data
from utils.functions.validate_natural_key import validate_natural_key_df

<h5> Configuración de la aplicación de Spark </h5>

In [0]:
spark = (
    SparkSession.builder
    .appName('PipelineSegmentSummaryGoldApp')
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

In [0]:
table_name = 'gold.sales_summarized.customer_segmentation'
output_table_name = 'gold.sales_summarized.segment_summary'
customer_segmentation_final = spark.table(table_name).drop('CREATED_AT', 'UPDATED_AT', 'DELETED', 'DELETED_AT','PK_HASH','R_HASH')

In [0]:
segment_summary = (
    customer_segmentation_final
    .groupBy("customer_segment")
    .agg(
        count("IDCLIENTE").alias("customer_count"),
        avg("monetary_value").alias("avg_monetary_value"),
        avg("frequency").alias("avg_frequency"),
        avg("recency_days").alias("avg_recency_days"),
        spark_sum("monetary_value").alias("total_revenue"),
        avg("purchase_frequency_rate").alias("avg_purchase_frequency_rate")
    )
    .withColumn("avg_monetary_value", spark_round(col("avg_monetary_value"), 2))
    .withColumn("avg_frequency", spark_round(col("avg_frequency"), 2))
    .withColumn("avg_recency_days", spark_round(col("avg_recency_days"), 2))
    .withColumn("total_revenue", spark_round(col("total_revenue"), 2))
    .withColumn("avg_purchase_frequency_rate", spark_round(col("avg_purchase_frequency_rate"), 2))
    .orderBy(col("total_revenue").desc())
)

In [0]:
transformed_df = prepare_system_dataframe(customer_segmentation_final, ['IDCLIENTE'])

In [0]:
upsert_data(transformed_df,output_table_name)